In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Além de [visualizar instruções em um Circuit](/guides/visualize-circuits), você pode querer visualizar o agendamento de um Circuit usando o método [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) do Qiskit. Essa visualização pode ajudar você a identificar rapidamente o tempo ocioso nos Qubits, por exemplo. No entanto, esse método não retorna resultados precisos para circuits dinâmicos. Para visualizar o agendamento de circuits dinâmicos, use o método `draw_circuit_schedule_timing`, conforme descrito na seção de [suporte ao Qiskit Runtime](#qr-support).

## Exemplos

Para visualizar um programa de Circuit agendado, você pode chamar essa função com um conjunto de argumentos de controle. A maior parte da aparência da imagem de saída pode ser modificada por uma folha de estilo, mas isso não é obrigatório.

### Desenhar com a folha de estilo padrão

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Desenhar com uma folha de estilo adequada para depuração de programas

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Você pode criar funções personalizadas de gerador ou layout e atualizar uma folha de estilo existente com essas funções personalizadas. Dessa forma, você pode controlar a maior parte da aparência da imagem de saída sem modificar a base de código do visualizador de Circuit agendado. Consulte a referência da API do [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) para mais exemplos.
<span id="qr-support"></span>
## Suporte ao Qiskit Runtime
Embora o visualizador de linha do tempo integrado ao Qiskit seja útil para circuits estáticos, ele pode não refletir com precisão o tempo de execução de [circuits dinâmicos](/guides/classical-feedforward-and-control-flow) devido a operações implícitas, como broadcasting e determinação de branch. Como parte do suporte a circuits dinâmicos, o Qiskit Runtime retorna as informações precisas de tempo do Circuit dentro dos resultados do job quando solicitado.

> **Note:** - Esta é uma função experimental. Ela está em status de versão prévia e, portanto, sujeita a alterações.
> - Esta função se aplica somente a jobs do Qiskit Runtime Sampler.
> - Embora o tempo total do Circuit seja retornado nos metadados de "compilação", esse NÃO é o tempo usado para cobrança (quantum time).
### Habilitar a recuperação de dados de tempo
Para habilitar a recuperação de dados de tempo, defina o sinalizador experimental `scheduler_timing` como `True` ao executar o job do primitivo.